# Lab 04: Optimization and Regularization

            **Duration:** 3 hours  
            **Lecture alignment:** Week 4 — Optimizers, scheduling, normalization, and regularization  
            **CLO mapping:** CLO-2, CLO-3, CLO-4  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Compare SGD, momentum, RMSprop, and Adam under a matched budget.
- Apply normalization, dropout, weight decay, scheduling, and early stopping.
- Select and restore a model using validation rather than test results.

            ## Three-hour activity plan

            - 0–30 min: data splits and leakage audit
- 30–90 min: optimizer benchmark
- 90–125 min: scheduler and normalization
- 125–155 min: dropout, weight decay, and early stopping
- 155–180 min: checkpoint, plots, and recommendation


## Book grounding

            - Goodfellow, Bengio, and Courville, *Deep Learning*, MIT Press, 2016.
- Zhang, Lipton, Li, and Smola, *Dive into Deep Learning*, Cambridge University Press, 2024.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20264
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_04")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_04"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 4, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Which optimizer will reduce training loss fastest, and will that necessarily identify the best regularized test model?

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — A controlled optimization and regularization benchmark


In [ ]:
def make_clusters(n=800):
    centers = torch.tensor([[-1.3,-1.0], [-1.3,1.0], [1.3,-1.0], [1.3,1.0]])
    labels = torch.randint(0, 4, (n,)); X = centers[labels] + .55*torch.randn(n, 2)
    order = torch.randperm(n); return X[order], labels[order]
X, y = make_clusters(650 if FAST_MODE else 2600)
ntr, nv = int(.55*len(X)), int(.2*len(X))
Xtr, Xv, Xte = X[:ntr], X[ntr:ntr+nv], X[ntr+nv:]
ytr, yv, yte = y[:ntr], y[ntr:ntr+nv], y[ntr+nv:]

class RegularizedMLP(nn.Module):
    def __init__(self, dropout=0.0, normalization=False):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 48), nn.LayerNorm(48) if normalization else nn.Identity(),
                                 nn.ReLU(), nn.Dropout(dropout), nn.Linear(48, 48), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(48, 4))
    def forward(self, x): return self.net(x)

optimizer_factories = {
    "SGD": lambda p: torch.optim.SGD(p, lr=.12),
    "Momentum": lambda p: torch.optim.SGD(p, lr=.08, momentum=.9),
    "RMSprop": lambda p: torch.optim.RMSprop(p, lr=.012),
    "Adam": lambda p: torch.optim.Adam(p, lr=.02),
}
def fit(name, dropout=0.0, weight_decay=0.0, normalization=False):
    torch.manual_seed(SEED); model = RegularizedMLP(dropout, normalization).to(DEVICE)
    opt = optimizer_factories[name](model.parameters())
    for group in opt.param_groups: group["weight_decay"] = weight_decay
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=35 if FAST_MODE else 100)
    train_hist, val_hist = [], []; best = (float("inf"), None); stale = 0
    for epoch in range(35 if FAST_MODE else 100):
        model.train(); opt.zero_grad(); loss = F.cross_entropy(model(Xtr.to(DEVICE)), ytr.to(DEVICE)); loss.backward(); opt.step(); scheduler.step()
        model.eval()
        with torch.no_grad(): val = F.cross_entropy(model(Xv.to(DEVICE)), yv.to(DEVICE)).item()
        train_hist.append(loss.item()); val_hist.append(val)
        if val < best[0]-1e-4:
            best = (val, {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}); stale = 0
        else: stale += 1
        if stale >= 8: break
    model.load_state_dict(best[1]); model.eval()
    with torch.no_grad(): acc = (model(Xte.to(DEVICE)).argmax(1).cpu()==yte).float().mean().item()
    return model, train_hist, val_hist, acc

optimizer_results = {name: fit(name) for name in optimizer_factories}
regularized = fit("Adam", dropout=.25, weight_decay=1e-3, normalization=True)
metrics = {name: {"epochs": len(v[1]), "test_accuracy": v[3], "best_val_loss": min(v[2])}
           for name, v in optimizer_results.items()}
metrics["Adam+regularization"] = {"epochs": len(regularized[1]), "test_accuracy": regularized[3], "best_val_loss": min(regularized[2])}
print(json.dumps(metrics, indent=2))


## Activity 2 — Learning curves, checkpointing, and augmentation reasoning


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7))
for name, (_, train_hist, val_hist, _) in optimizer_results.items():
    axes[0].plot(train_hist, label=name); axes[1].plot(val_hist, label=name)
axes[0].set(title="Training loss", xlabel="epoch"); axes[1].set(title="Validation loss", xlabel="epoch")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8); fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "optimization_regularization.png", dpi=150); plt.show()
best_name = max(optimizer_results, key=lambda key: optimizer_results[key][3])
torch.save(optimizer_results[best_name][0].state_dict(), ARTIFACT_DIR / "best_model.pt")
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
# For these 2-D points, small Gaussian jitter is the analogue of a label-preserving augmentation.
jittered = Xtr + .04*torch.randn_like(Xtr)
assert jittered.shape == Xtr.shape


## Automated checks


In [ ]:
assert set(optimizer_results) == {"SGD", "Momentum", "RMSprop", "Adam"}
assert all(np.isfinite(v[1]).all() and np.isfinite(v[2]).all() for v in optimizer_results.values())
assert max(v[3] for v in optimizer_results.values()) > .80
assert (ARTIFACT_DIR / "best_model.pt").exists() and (ARTIFACT_DIR / "metrics.json").exists()
print("All Lab 04 checks passed.")


## Deliverables

                - Matched optimizer comparison
- Regularization/early-stopping result
- Best checkpoint, metrics JSON, and learning-curve figure

                Submit the executed notebook and the files created in `/content/artifacts/lab_04/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    print("Extension: inject 15% label noise and repeat the baseline/regularized comparison.")
else:
    print("Extension disabled: test robustness to label noise and calibrate predicted probabilities.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
